# Sequential agent handoff

This lab sends a network specialist's finding to an evidence reviewer as an explicit, inspectable message. The agents do not automatically share a conversation.

![Sequential handoff workflow](figures/sequential-handoff-workflow.svg)

The original case is sent to the network specialist first. Its finding becomes the explicit `network_handoff` message. The evidence reviewer receives both that message and the original case, so it can check the report against the source materials.

## Step 1: Import AgentScope and configuration helpers

This cell imports the agent, message, model, and local-configuration classes used to build the two-stage workflow.

In [ ]:
import os

from dotenv import load_dotenv
from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel


## Step 2: Configure the shared model connection

Both agents use this one model object. Reusing the connection does not merge their prompts or their short-term states.

In [ ]:
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")

if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=170),
)


## Step 3: Create the first and second specialists

The network specialist describes observed connections. The evidence reviewer checks supported facts and missing evidence after it receives the handoff.

In [ ]:
network_specialist = Agent(
    name="network_specialist",
    system_prompt=(
        "You are the first specialist in a practice security workflow. "
        "Report what the network alert observed and what the alert cannot show."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)

evidence_reviewer = Agent(
    name="evidence_reviewer",
    system_prompt=(
        "You are the second specialist in a practice security workflow. "
        "Use the original case and the supplied network report. "
        "List supported facts, missing evidence, and one next item to collect."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)


## Step 4: Define the original practice case

Both stages can see these original materials. Including them again in the second stage lets the reviewer check the handoff rather than relying on it alone.

In [ ]:
case_materials = """
Practice case INC-204
- Network-monitoring alert: at 09:14 UTC, an automated sensor recorded a workstation making forty-three outbound contacts to 192.0.2.44.
- The local practice list marks 192.0.2.44 as suspicious and says this requires analyst review; it is not proof of malicious activity.
- The alert does not identify the process, user action, payload, or destination port/service for the contacts.
""".strip()

print(case_materials)


## Step 5: Run the first specialist

The first request contains only the original case. The helper extracts readable text from its final response for the handoff.

In [ ]:
def response_text(response: Msg) -> str:
    """Extract readable text from an AgentScope response message."""
    return "".join(block.text for block in response.content if isinstance(block, TextBlock))

network_request = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text=(
        "Review this practice case in two concise bullets.\n\n"
        f"{case_materials}"
    ))],
)

network_response = await network_specialist.reply(network_request)
network_finding = response_text(network_response)
print(network_finding)


## Step 6: Create and inspect the handoff message

This is the explicit transfer point. `network_handoff` is ordinary structured data created by the notebook, not an automatically shared agent memory.

In [ ]:
network_handoff = Msg(
    name=network_specialist.name,
    role="user",
    content=[TextBlock(text=(
        "Network specialist handoff. Treat this as a report to check against the original case, not as proof:\n"
        f"{network_finding}"
    ))],
)

print(network_handoff.get_text_content())


## Step 7: Give the original case and handoff to the reviewer

The second request contains both sources. Its response should preserve uncertainty and recommend evidence to collect rather than reach a final verdict.

In [ ]:
review_request = Msg(
    name="workflow",
    role="user",
    content=[TextBlock(text=(
        "Original practice case:\n"
        f"{case_materials}\n\n"
        f"{network_handoff.get_text_content()}\n\n"
        "Review the evidence in three concise bullets: supported facts, missing evidence, and one next item to collect."
    ))],
)

review_response = await evidence_reviewer.reply(review_request)
print(response_text(review_response))


## Step 8: Checkpoint

Temporarily change the network specialist's prompt to make an overly confident claim. Then rerun the workflow and verify that the evidence reviewer uses the original case to preserve the alert's limits. Restore the careful prompt afterward.